# Document parsing with HunyuanOCR and OpenVINO

[HunyuanOCR](https://huggingface.co/tencent/HunyuanOCR) is a lightweight, end-to-end
OCR-specialised vision-language model released by the Tencent Hunyuan team. It pairs a
SigLIP-style vision encoder with a compact HunYuan text decoder and unifies **document
parsing**, **text spotting**, **information extraction**, and **text-image translation**
in a single end-to-end VLM, while staying small enough for on-device deployment.

In this tutorial we convert HunyuanOCR to OpenVINO IR via optimum-intel, optionally
compress its weights to **INT8** (or keep **FP16**) with [NNCF](https://github.com/openvinotoolkit/nncf),
and run inference on Intel CPU / integrated GPU / Arc GPU. The basic inference usage is
aligned with the official HuggingFace `transformers` (native) snippet from the
[model card](https://huggingface.co/tencent/HunyuanOCR). A streaming Gradio demo — with
the same task presets as the official [HunyuanOCR Space](https://huggingface.co/spaces/tencent/HunyuanOCR) —
is provided as well.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
  - [Select precision](#Select-precision)
- [Run OpenVINO model](#Run-OpenVINO-model)
  - [Select inference device](#Select-inference-device)
- [OCR inference](#OCR-inference)
- [Interactive demo](#Interactive-demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is
using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/hunyuan-ocr/hunyuan-ocr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

HunyuanOCR ships an official `transformers` integration (`HunYuanVLForConditionalGeneration`
+ `AutoProcessor`) that requires `transformers>=5.13`, and relies on the `hunyuan-ocr-support`
branch of the [openvino-dev-samples/optimum-intel](https://github.com/openvino-dev-samples/optimum-intel)
fork for the OpenVINO export.

In [ ]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    Path("cmd_helper.py").write_text(r.text, encoding="utf-8")

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    Path("notebook_utils.py").write_text(r.text, encoding="utf-8")

# gradio_helper.py ships alongside this notebook; no download needed when
# running from the repository checkout.

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("hunyuan-ocr.ipynb")

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx

from notebook_utils import pip_install

pip_install("-q", "torch>=2.8", "torchvision", "--extra-index-url", "https://download.pytorch.org/whl/cpu")
pip_install("-q", "git+https://github.com/openvino-dev-samples/optimum-intel.git@hunyuan-ocr-support")
pip_install("-q", "openvino>=2026.2.0", "nncf>=2.17.0")
pip_install("-q", "gradio>=5.25.0", "Pillow>=10")
# HunyuanOCR needs the transformers release that introduces the "hunyuan_vl" architecture.
# Use an exact pin (no ">" character) so it is installed reliably on Windows CI: with the
# pip_install helper's shell=True on Windows, ">=" is interpreted by cmd.exe as an output
# redirection and the version constraint gets silently dropped, which lets pip resolve to an
# older transformers (capped by optimum-intel's transformers<5.14) that lacks "hunyuan_vl".
pip_install("-q", "transformers==5.13.1")

In [ ]:
# The demo/sample document image is generated on the fly (no binary asset is
# shipped with the notebook), so this notebook is fully self-contained.
from gradio_helper import make_sample_image

sample_image = make_sample_image()
print("Sample image saved to:", sample_image)

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

[Optimum Intel](https://huggingface.co/docs/optimum/intel/index) exposes a command-line
interface to export HuggingFace models to OpenVINO IR. For HunyuanOCR we use the
`image-text-to-text` task, which produces three IR submodels: the vision embeddings
(patch embedder + vision transformer + merger), the text-embedding lookup, and the
stateful language-model decoder with the tied LM head.

```bash
optimum-cli export openvino --model tencent/HunyuanOCR --task image-text-to-text --weight-format int8 HunyuanOCR/INT8
```

### Select precision
[back to top ⬆️](#Table-of-contents:)

The language decoder dominates the footprint, so **INT8** weight compression roughly halves
the model size relative to **FP16** with negligible quality loss on OCR tasks. Pick a
precision below.

In [ ]:
import ipywidgets as widgets

model_id = "tencent/HunyuanOCR"

precision = widgets.Dropdown(
    options=["INT8", "FP16"],
    value="INT8",
    description="Precision:",
    disabled=False,
)
precision

In [ ]:
from pathlib import Path
from cmd_helper import optimum_cli

model_base_dir = Path(model_id.split("/")[-1])  # "HunyuanOCR"
additional_args = {"task": "image-text-to-text"}

if precision.value == "INT8":
    model_dir = model_base_dir / "INT8"
    additional_args["weight-format"] = "int8"
else:
    model_dir = model_base_dir / "FP16"
    additional_args["weight-format"] = "fp16"

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

print("OpenVINO IR ready at:", model_dir)

## Run OpenVINO model
[back to top ⬆️](#Table-of-contents:)

optimum-intel's `OVModelForVisualCausalLM` loads the IR we just exported and exposes a
transformers-compatible API: `.generate()`, HuggingFace chat templates, token streamers, etc.
The usage below mirrors the native `transformers` snippet from the HunyuanOCR model card.

### Select inference device
[back to top ⬆️](#Table-of-contents:)

HunyuanOCR runs on Intel CPU, integrated GPUs and Arc GPUs. NPU inference is not currently
supported for image-text-to-text models and is excluded from the widget below.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
from transformers import AutoProcessor
from optimum.intel import OVModelForVisualCausalLM

processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device.value)

## OCR inference
[back to top ⬆️](#Table-of-contents:)

HunyuanOCR is driven by task-specific text prompts. The default **document parsing** prompt
extracts the body text as Markdown (tables as HTML, formulas as LaTeX) in reading order.
The code below mirrors the official snippet in the
[`tencent/HunyuanOCR` model card](https://huggingface.co/tencent/HunyuanOCR): build a chat
message with an image + text prompt, apply the chat template, and `generate` greedily.

In [ ]:
import sys
from pathlib import Path
from threading import Thread

from PIL import Image
from IPython.display import display
from transformers import TextIteratorStreamer

# Default HunyuanOCR document-parsing prompt.
DOC_PARSE_PROMPT = (
    "Extract all the body text from the document image in markdown format. "
    "Ignore headers and footers, express tables in HTML format and formulas "
    "in LaTeX format, and organize the output in reading order."
)


def run_hunyuan_ocr(image_path, prompt=DOC_PARSE_PROMPT, max_new_tokens=1024):
    """Run HunyuanOCR and stream the decoded text to stdout as it arrives.

    Returns the full decoded string once generation finishes.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path)},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        streamer=streamer,
    )
    thread = Thread(target=model.generate, kwargs=gen_kwargs, daemon=True)
    thread.start()

    output = ""
    for piece in streamer:
        sys.stdout.write(piece)
        sys.stdout.flush()
        output += piece
    thread.join(timeout=1.0)
    sys.stdout.write("\n")
    sys.stdout.flush()
    return output

In [ ]:
display(Image.open(sample_image))
_ = run_hunyuan_ocr(sample_image, DOC_PARSE_PROMPT)

In [ ]:
# The same model handles other OCR tasks by simply changing the prompt.
# Here we ask it to detect and localise the text (text spotting).
_ = run_hunyuan_ocr(sample_image, "Detect and recognize the text in the image, and output the text with its coordinates.", max_new_tokens=512)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Upload an image, pick a task preset (or type your own instruction), and watch the decoded
text stream in. The task presets mirror the official
[HunyuanOCR Space](https://huggingface.co/spaces/tencent/HunyuanOCR).

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, processor)

try:
    demo.launch(debug=False, height=800)
except Exception:
    demo.launch(debug=False, share=True, height=800)

# If you are launching remotely, specify server_name and server_port:
# demo.launch(server_name='your server name', server_port='server port number')